### Reproducing the Error for Solo v2 as found in Theodoridis et al. 2022

This error happens because Solo v2 does not predict bboxes by default, adding zeros to not break the implmementation. This happens here:

- https://github.com/aim-uofa/AdelaiDet/blob/master/adet/modeling/solov2/solov2.py#L496

In Theodoridis et al. 2022, the authors first eval with iou_type 'segm' and with the included zeros here:

- https://github.com/JohannesTheo/trapped-in-texture-bias/blob/main/src/detection/SOLO/solo_detection.py#L10

and then separately derive the bboxes here:

- https://github.com/JohannesTheo/trapped-in-texture-bias/blob/main/src/detection/SOLO/solo_derive_bbox.py

Since they use their custom save and eval method for segm and not the detectron 2 implementation, this will result in wrong values for small, medium and large objects as shown below.

Please note that this only affects area calculations. The default AP value is NOT affected. Note further that we only show the error for ms-coco here. In the paper we report the problem and its correction for ocs-coco, which consists of 30 sub-datasets, too many to include here.

Finally note that this error or problem is not documented properly in neither coco api, detectron 2 or mmdetection (only in the eval code)!

In [21]:
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

In [22]:
def run_coco_eval(ann_file, results_file, iou_type, remove_bbox=False):

    assert iou_type in ['bbox', 'segm']

    with open(results_file, 'r') as f:
        predictions = json.load(f)
        print("BBox sample:", predictions[0]['bbox'], '\n')

    if iou_type == 'segm' and remove_bbox:
        print("Removing 'bbox' field from predictions", '\n')

        # See: https://github.com/facebookresearch/detectron2/blob/main/detectron2/evaluation/coco_evaluation.py#L581
        # See: https://github.com/open-mmlab/mmdetection/blob/cfd5d3a985b0249de009b67d04f37263e11cdf3d/mmdet/evaluation/metrics/coco_metric.py#L454

        # Refer to https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/coco.py#L331  # noqa
        # When evaluating mask AP, if the results contain bbox,
        # cocoapi will use the box area instead of the mask area
        # for calculating the instance area. Though the overall AP
        # is not affected, this leads to different
        # small/medium/large mask AP results.
        for x in predictions:
            x.pop('bbox')

    cocoGt = COCO(ann_file)
    cocoDt = cocoGt.loadRes(predictions)
    cocoEval = COCOeval(cocoGt, cocoDt, iou_type)

    cocoEval.evaluate()
    cocoEval.accumulate()
    cocoEval.summarize()


In [ ]:
ann_file = "./annotations/coco/instances_val2017.json"
results_with_bbox = "./solo_v2/ms-coco_results_with_derived_bbox.json"
results_with_zeros = "./solo_v2/ms-coco_results_with_zero_bbox.json"

In [24]:
# Small error - since derived bbox have slightly different area than masks
run_coco_eval(ann_file, results_with_bbox, 'segm', remove_bbox=False)

BBox sample: [388.0, 69.0, 110.0, 276.0] 

loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.08s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=11.08s).
Accumulating evaluation results...
DONE (t=1.69s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.375
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.580
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.398
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.171
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.419
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.541
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.311
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.485
 Average Recall     (

In [25]:
# Large error - since zero bbox have no area at all - This happened in Theodoridis et al. 2022
run_coco_eval(ann_file, results_with_zeros, 'segm', remove_bbox=False)

BBox sample: [0.0, 0.0, 0.0, 0.0] 

loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.08s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=11.51s).
Accumulating evaluation results...
DONE (t=1.65s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.375
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.580
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.398
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.088
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.564
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.707
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.311
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.485
 Average Recall     (AR) @[ 

In [26]:
# This will produce the same correct eval, since bbox areas are removed
run_coco_eval(ann_file, results_with_bbox, 'segm', remove_bbox=True)
run_coco_eval(ann_file, results_with_zeros, 'segm', remove_bbox=True)

BBox sample: [388.0, 69.0, 110.0, 276.0] 

Removing 'bbox' field from predictions 

loading annotations into memory...
Done (t=0.19s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.30s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=12.88s).
Accumulating evaluation results...
DONE (t=1.71s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.375
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.580
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.398
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.155
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.411
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.569
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.311
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | ma